# 🌉 18 · From simulation to reality: domain shift, adaptation and synthetic data
### Why a network trained on simulated tomograms stumbles on real cells — and five ways to help it walk

**BiologyToResearchV3** · Part D — Deep learning on subtomograms (2017–2024) · Module 18 of 26

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearchV3/notebooks/18_simulation_to_reality.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 4–6 min to run everything); a GPU is optional |
| **Core frameworks** | Python 3, NumPy, SciPy, PyTorch, scikit-learn, scikit-image, Matplotlib |
| **Prerequisites** | Modules 05 (PyTorch), 06–08 (CTF, missing wedge, simulation), 15 (CNN classifiers), 17 (learning with few labels) |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Lin, Zeng *et al.* 2019 (3D-ADA); Che *et al.* 2019 (domain randomization ⚠️); Wu *et al.* 2019 (RAT ⚠️); Yu *et al.* 2021 (few-shot domain adaptation); Bandyopadhyay *et al.* 2022 (Cryo-Shift); Wu *et al.* 2022 (CryoETGAN); Li *et al.* 2025 (Vox-UDA) and its 2026 IJCV extension — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All experiments in this course are small, controlled *toy versions* of published methods. They are designed to teach
> the core idea of each paper; they do **not** reproduce the papers' numbers. Here we even work in **2D** (projection
> images) instead of 3D subtomograms so that every network trains in seconds on a CPU; the ideas carry over unchanged.
> Molecular densities come from real PDB entries (via the PolNet simulator); offline, the notebook falls back to simple
> stand-in shapes and says so.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Define** *domain shift* (source vs target distribution, covariate shift) and **measure** it three ways: accuracy drop,
   power spectra, and a "can a classifier tell the domains apart?" test.
2. **Build** a labelled *source* domain (a clean simulator) and an unlabelled *target* domain (a different microscope,
   noise colour, missing wedge and crowding) from the molecular zoo.
3. **Implement** domain randomization and explain when it works (the target must lie inside the randomized ranges).
4. **Implement** adversarial domain adaptation with a gradient-reversal layer (DANN) and **explain** how 3D-ADA and
   Cryo-Shift use the same min–max idea.
5. **Use** a handful of labelled target images (few-shot domain adaptation) together with the Deep CORAL loss.
6. **Explain** unpaired image translation (CycleGAN → CryoETGAN) and **train** a tiny physics-constrained "learned
   microscope" with a Wasserstein critic.
7. **Implement** a 2D Vox-UDA-lite for segmentation — a noise generation module plus mean-teacher pseudo-labels on
   bilateral-filtered targets — and **measure** the target Dice step by step.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | check the computer, import tools, load the zoo | 5 min |
| 1 · Two microscopes, one cell | build a source and a target domain; compare their spectra | 20 min |
| 2 · The accuracy drop | train on source, test on target; see the gap in feature space | 15 min |
| 3 · Domain randomization | train on many random microscopes | 15 min |
| 4 · Adversarial adaptation | gradient reversal (DANN), 3D-ADA, Cryo-Shift | 25 min |
| 5 · A few labelled targets | few-shot fine-tuning + Deep CORAL | 15 min |
| 6 · Learning the microscope | unpaired translation, CryoETGAN, a tiny WGAN | 20 min |
| 7 · Segmentation: Vox-UDA-lite | noise generation + denoised pseudo-labels | 25 min |
| 8 · Summary | which tool for which situation | 5 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Imagine you want to count ribosomes and proteasomes in tomograms of rat neurons. Labelling thousands of real
subtomograms by hand would take months, so you do the sensible thing: you **simulate** subtomograms from PDB structures
(Module 08), where every label is known for free, and train a 3D CNN on them (Module 15). On simulated test data the
network is superb. On the real neurons it is barely better than guessing.

Nothing is wrong with the biology — a ribosome is a ribosome. What changed is **how the images were made**: a different
defocus and contrast transfer function, a different tilt range, detector noise that is not white, thicker ice,
neighbours crowding into the box, and molecules that flex. The network learned shortcuts that only hold in the
simulator. This is called **domain shift**, and it is one of the main reasons deep learning was slow to reach real
cellular cryo-ET. Between 2019 and 2026 the Xu Lab published a whole family of remedies; this notebook builds a small,
working version of each.

---
## 0 · Environment setup

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook is designed for a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed yet (the next cell installs it).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn"), ("skimage", "scikit-image"), ("torch", "torch")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import copy
import os
import struct
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy import ndimage
from skimage.restoration import denoise_bilateral
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # everything here also runs fine on the CPU
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| PyTorch", torch.__version__, "| device:", DEVICE, "| seed", SEED)

### 🧰 Course toolkit

The helpers below are copied verbatim from [`course_toolkit.py`](../course_toolkit.py): downloading the zoo, rotating
volumes (Module 02), Fourier masks and the missing wedge (Modules 03, 07) and the contrast transfer function (Module 06).
Just run the cell.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
POLNET_COMMIT = "73e613cc9162d26c7fd740ce3299bcb280a3e3b6"
POLNET_URL = ("https://raw.githubusercontent.com/anmartinezs/polnet/"
              + POLNET_COMMIT + "/data/templates/{kind}/{name}.{ext}")
CACHE_DIR = os.environ.get("CRYO_COURSE_CACHE", "course_data")

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2277, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=5851, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3325, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=659, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=839, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1309, symmetry="C1"),
    # NB: PolNet's 4V94 map holds TWO TRiC complexes side by side (1.95 MDa ≈ 2 × ~1 MDa).
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1952, symmetry="C2"),
    "1bxn": dict(name="RuBisCO", organism="Cupriavidus necator", mass_kDa=280, symmetry="D4"),
    "1qvr": dict(name="ClpB chaperone", organism="Thermus thermophilus", mass_kDa=297, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=None, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=None, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=239, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=236, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=43, symmetry="C1"),
}


def _download(url: str, dest: str, timeout: float = 60.0) -> bool:
    """Download ``url`` to ``dest`` (no-op if it exists). Returns True on success."""
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        return True
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r, open(dest + ".part", "wb") as f:
            f.write(r.read())
        os.replace(dest + ".part", dest)
        return True
    except Exception as exc:  # network blocked, offline, ...
        print(f"  ⚠️  download failed for {url.split('/')[-1]}: {type(exc).__name__}")
        return False


def read_mrc(path_or_bytes) -> tuple[np.ndarray, float]:
    """Minimal MRC2014 reader (modes 0, 1, 2, 6). Returns (array[z, y, x], voxel_size_Å).

    The MRC header is 1024 bytes of little-endian 32-bit words; words 1–3 are the
    grid size (nx, ny, nz), word 4 the data type, words 11–13 the cell size in Å,
    and word 24 the length of an optional extended header.
    """
    raw = path_or_bytes if isinstance(path_or_bytes, (bytes, bytearray)) else open(path_or_bytes, "rb").read()
    if raw[:2] == b"\x1f\x8b":  # gzip
        import gzip
        raw = gzip.decompress(raw)
    nx, ny, nz, mode = struct.unpack("<4i", raw[0:16])
    mx, my, mz = struct.unpack("<3i", raw[28:40])
    cella = struct.unpack("<3f", raw[40:52])
    mapc, mapr, maps = struct.unpack("<3i", raw[64:76])
    nsymbt = struct.unpack("<i", raw[92:96])[0]
    dtype = {0: np.int8, 1: np.int16, 2: np.float32, 6: np.uint16}[mode]
    data = np.frombuffer(raw, dtype=dtype, offset=1024 + nsymbt, count=nx * ny * nz)
    vol = data.reshape(nz, ny, nx).astype(np.float32)
    if (mapc, mapr, maps) != (1, 2, 3) and (mapc, mapr, maps) != (0, 0, 0):
        # reorder axes so that the array is [z, y, x]
        order = [3 - maps, 3 - mapr, 3 - mapc]
        vol = np.transpose(vol, np.argsort(order))
    voxel = float(cella[0] / mx) if mx > 0 else 1.0
    return vol, voxel


def fetch_zoo_map(pdb_id: str, voxel: str = "10A") -> tuple[np.ndarray, float, str]:
    """Return (density[z, y, x], voxel_size_Å, source) for a molecular-zoo entry.

    Tries the pinned PolNet template first; if the download fails (no internet),
    falls back to a *procedural* stand-in so the notebook still runs.
    """
    pdb_id = pdb_id.lower()
    dest = os.path.join(CACHE_DIR, f"polnet_{voxel}_{pdb_id}.mrc")
    url = POLNET_URL.format(kind=f"mrcs_{voxel}", name=pdb_id, ext="mrc")
    if _download(url, dest):
        vol, vs = read_mrc(dest)
        return vol, vs, f"PolNet template from PDB {pdb_id.upper()}"
    vol = procedural_molecule(pdb_id)
    return vol, 10.0, f"procedural stand-in for {pdb_id.upper()} (offline fallback)"


def procedural_molecule(pdb_id: str, size: int = 40) -> np.ndarray:
    """Offline fallback: a coarse shape with the right size/symmetry flavour (1 nm voxels)."""
    g = (np.arange(size) - size / 2 + 0.5)
    z, y, x = np.meshgrid(g, g, g, indexing="ij")
    r = np.sqrt(x**2 + y**2)

    def blob(cz, cy, cx, s):
        return np.exp(-((z - cz)**2 + (y - cy)**2 + (x - cx)**2) / (2 * s**2))
    pid = pdb_id.lower()
    if pid in ("4v4r", "4v7r", "5mrc"):          # two unequal subunits
        v = blob(-3, 0, 0, 7.5) + 0.8 * blob(6, 2, 1, 5.5)
    elif pid in ("3j9i", "6utj", "4cr2"):        # stacked rings / barrel
        v = np.exp(-((r - 5.5)**2) / 4.0) * (np.abs(z) < 7.5)
        if pid != "3j9i":
            v = v + 0.8 * blob(10, 0, 0, 4)
    elif pid == "4v94":                          # double ring with cavity
        v = np.exp(-((r - 7)**2) / 6.0) * (np.abs(z) < 8)
    elif pid in ("3cf3", "1qvr"):                # hexameric ring
        ang = np.arctan2(y, x)
        v = np.exp(-((r - 5)**2) / 3.0) * (np.abs(z) < 4) * (1 + 0.5 * np.cos(6 * ang))
    elif pid == "1bxn":                          # compact ball
        v = blob(0, 0, 0, 4.5)
    else:                                        # small asymmetric blob
        v = blob(0, 0, 0, 3) + 0.5 * blob(2, 2, 0, 2)
    return ndimage.gaussian_filter(v.astype(np.float32), 1.0)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    PolNet templates are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    com = np.array(ndimage.center_of_mass(v))
    shift = np.array(v.shape) / 2 - 0.5 - com
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)
    dst0 = np.maximum((size - s) // 2, 0)
    n = np.minimum(s, size)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Fetch and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = fetch_zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)


def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def freq_grid(shape):
    """Frequency coordinates (cycles/voxel) for an unshifted FFT; returns list per axis."""
    return np.meshgrid(*[np.fft.fftfreq(n) for n in shape], indexing="ij")


def radial_frequency(shape) -> np.ndarray:
    """|k| in cycles/voxel for every coefficient of an unshifted FFT."""
    return np.sqrt(sum(k**2 for k in freq_grid(shape)))


def missing_wedge_mask(shape, tilt_max_deg: float = 60.0) -> np.ndarray:
    """Binary Fourier mask of a single-axis tilt series about y (±tilt_max).

    A coefficient (k_z, k_y, k_x) is measured if |k_z| ≤ |k_x|·tan(tilt_max)
    — the formula M(ξ) = 1(|ξ_3| ≤ |ξ_1| tan θ) used by Xu et al. (2012).
    For 2D images [z, x] pass a 2-tuple shape.
    """
    k = freq_grid(shape)
    kz, kx = k[0], k[-1]
    m = np.abs(kz) <= np.abs(kx) * np.tan(np.radians(tilt_max_deg)) + 1e-9
    m[(0,) * len(shape)] = True               # keep the mean (DC) term
    return m.astype(np.float32)


def apply_fourier_mask(vol: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Multiply the FFT of ``vol`` by ``mask`` (unshifted layout) and return the real part."""
    return np.real(np.fft.ifftn(np.fft.fftn(vol) * mask)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def electron_wavelength_A(kV: float = 300.0) -> float:
    """Relativistic electron wavelength in Å (0.0197 Å at 300 kV)."""
    V = kV * 1e3
    return 12.2643 / np.sqrt(V * (1 + 0.978466e-6 * V))


def ctf_2d(shape, pixel_A: float, defocus_um: float = 3.0, kV: float = 300.0,
           Cs_mm: float = 2.7, amp_contrast: float = 0.07, B: float = 0.0) -> np.ndarray:
    """Contrast transfer function on an unshifted FFT grid (underfocus is positive).

    CTF(k) = −[√(1−A²)·sin χ(k) + A·cos χ(k)]·exp(−B k²/4),
    χ(k) = π λ Δf k² − (π/2) C_s λ³ k⁴, with k in 1/Å.
    """
    lam = electron_wavelength_A(kV)
    k2 = sum(kk**2 for kk in freq_grid(shape)) / pixel_A**2
    dz = defocus_um * 1e4
    cs = Cs_mm * 1e7
    chi = np.pi * lam * dz * k2 - 0.5 * np.pi * cs * lam**3 * k2**2
    ctf = -(np.sqrt(1 - amp_contrast**2) * np.sin(chi) + amp_contrast * np.cos(chi))
    return (ctf * np.exp(-B * k2 / 4)).astype(np.float32)


print("🧰 toolkit loaded: fetch_zoo_map, load_zoo, rotate_volume, missing_wedge_mask, ctf_2d, ...")

### Data: the molecular zoo

We use the four "star" molecules of the course: the 70S ribosome (4V4R), the 20S proteasome (3J9I), the singly capped
proteasome (6UTJ) and RuBisCO (1BXN), as density maps at 10 Å per voxel.

In [ ]:
# ── Load the four star molecules ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
NAMES = [ZOO[p]["name"] for p in CAST]
zoo = load_zoo(CAST, size=32, sigma=1.0)
print("volumes:", {p: v.shape for p, v in zoo.items()})

---
## 1 · Two microscopes, one cell

> **📍 Where we are.** In Module 15 you trained 3D CNNs on *simulated* subtomograms and in Module 17 you learned to cope
> with few labels. Both assumed that training and test images come from the **same** imaging process. This module drops
> that assumption: what happens when the test images come from a different "microscope", and what can we do about it
> without labelling the new data?

### 1.1 What is a domain?

Call an image $x$ and its label $y$ (ribosome, proteasome, …). A **domain** is the joint probability distribution
$P(x, y)$ of images and labels produced by one imaging pipeline.

- The **source domain** $S$: our simulator. We can make unlimited labelled pairs $(x, y)\sim P_S$.
- The **target domain** $T$: the real microscope. We get many images $x\sim P_T$, but **no labels** (or very few).

**Domain shift** means $P_S \neq P_T$. The most common kind in cryo-ET is **covariate shift**:

$$P_S(y \mid x) = P_T(y \mid x) \quad\text{but}\quad P_S(x) \neq P_T(x).$$

*In words:* the rule "this shape is a ribosome" is the same in both worlds, but the *pictures* look different (other
noise, blur, wedge, neighbours), so a network that learned the rule from source pictures may not recognise target pictures.

💡 A classic result (Ben-David *et al.*, 2010) bounds the error of a classifier $h$ on the target:

$$\varepsilon_T(h) \;\le\; \varepsilon_S(h) \;+\; \tfrac12\, d_{\mathcal H\Delta\mathcal H}(S, T) \;+\; \lambda .$$

*In words:* target error ≤ source error + **how easily the two domains can be told apart** (by classifiers of the kind we
use) + a term $\lambda$ that is small when one classifier could do well on both. Every method in this notebook attacks the
middle term — either by making the source *broader* (randomization, learned simulators) or by making the network's
features *unable to tell the domains apart* (adversarial alignment).

### 1.2 The clean images

To keep training fast we work with **2D projection images**: each molecule is rotated randomly in 3D and summed along
$y$, giving a 32 × 32-pixel image in the $(z, x)$ plane at 10 Å/pixel — exactly what one tilt image of an isolated
particle would record. (The papers did the same experiments with 3D subtomograms; nothing conceptual changes.)

In [ ]:
# ── 1.2 A pool of clean projections (600 per molecule) ──
def random_projections(volume: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    """Project ``n`` randomly rotated copies of ``volume`` along y → images [n, z, x]."""
    return np.stack([rotate_volume(volume, random_rotation(rng)).sum(axis=1) for _ in range(n)])


t0 = time.time()
N_PER_CLASS = 600
clean = np.concatenate([random_projections(zoo[p], N_PER_CLASS, rng) for p in CAST]).astype(np.float32)
clean /= clean.max()
labels = np.repeat(np.arange(len(CAST)), N_PER_CLASS)
print(f"{len(clean)} clean projections of shape {clean.shape[1:]} in {time.time() - t0:.1f} s")

### 1.3 Two imaging pipelines

A function `microscope(...)` turns a clean projection into a recorded image. Each step is something you met in Part B:

| Step | Parameter | Physics (module) |
|---|---|---|
| neighbours crowd into the box | `neighbour` (fraction of a second particle's density) | crowding (08) |
| contrast transfer function | `defocus_um`, B-factor `B` (Å²) | CTF (06) |
| missing wedge | `tilt_max` (°) | tomography (07) |
| noise, white or "coloured" | `snr`, `noise_beta` | statistics (04) |

**Coloured noise** has a power spectrum $\propto 1/|k|^{\beta}$: with $\beta = 0$ it is white (every frequency equally
noisy, as in most simple simulators), with $\beta > 0$ it is stronger at low frequencies — closer to what detectors,
thick ice and imperfect reconstruction produce in real tomograms.

In [ ]:
# ── 1.3 The imaging model ──
def coloured_noise(shape, rng: np.random.Generator, beta: float) -> np.ndarray:
    """Gaussian noise with power spectrum ∝ 1/|k|^beta (beta = 0 → white), unit variance."""
    k = radial_frequency(shape)
    k[(0,) * len(shape)] = 1.0
    amplitude = k ** (-beta / 2)
    amplitude[(0,) * len(shape)] = 0.0
    n = np.real(np.fft.ifftn(np.fft.fftn(rng.normal(size=shape)) * amplitude))
    return n / n.std()


def microscope(p: np.ndarray, rng: np.random.Generator, defocus_um=2.5, B=0.0, tilt_max=60.0, snr=0.5,
               noise_beta=0.0, neighbour=0.0, pool=None, pixel_A=10.0) -> np.ndarray:
    """Turn a clean 2D projection into a recorded image: neighbour → CTF → wedge → noise → normalise."""
    img = p.copy()
    if neighbour > 0 and pool is not None:          # part of another particle enters from a random side
        q = pool[rng.integers(len(pool))]
        dy, dx = rng.choice([-1, 1], 2) * rng.integers(18, 24, 2)
        img = img + neighbour * np.roll(q, (dy, dx), axis=(0, 1))
    img = apply_fourier_mask(img, -ctf_2d(img.shape, pixel_A, defocus_um=defocus_um, B=B))  # sign flip: dense = bright
    img = apply_fourier_mask(img, missing_wedge_mask(img.shape, tilt_max))
    sigma = np.sqrt(img.var() / snr)
    noise = coloured_noise(img.shape, rng, noise_beta) if noise_beta > 0 else rng.normal(size=img.shape)
    return normalize(img + sigma * noise)


SOURCE = dict(defocus_um=2.5, B=0.0, tilt_max=60.0, snr=0.5, noise_beta=0.0, neighbour=0.0)   # our simulator
TARGET = dict(defocus_um=5.0, B=150.0, tilt_max=50.0, snr=0.5, noise_beta=0.8, neighbour=0.5)  # the "real" microscope

# split the clean pool into four disjoint groups of particles
perm = rng.permutation(len(clean))
n = len(clean)
idx_src_train, idx_src_test = perm[: 3 * n // 8], perm[3 * n // 8: n // 2]
idx_tgt_train, idx_tgt_test = perm[n // 2: 7 * n // 8], perm[7 * n // 8:]


def make_domain(indices, params, rng):
    """Image the clean projections ``indices`` with one set of microscope ``params`` → (images, labels)."""
    X = np.stack([microscope(clean[i], rng, pool=clean, **params) for i in indices]).astype(np.float32)
    return X, labels[indices]


Xs, ys = make_domain(idx_src_train, SOURCE, rng)          # labelled source (training)
Xs_test, ys_test = make_domain(idx_src_test, SOURCE, rng)  # source test
Xt, yt_hidden = make_domain(idx_tgt_train, TARGET, rng)    # UNLABELLED target (we never use yt_hidden for training)
Xt_test, yt_test = make_domain(idx_tgt_test, TARGET, rng)  # target test — the answer key, for evaluation only
print(f"source train {Xs.shape}, source test {Xs_test.shape}, target (unlabelled) {Xt.shape}, target test {Xt_test.shape}")

In [ ]:
# ── 1.4 Look at both domains and compare their power spectra ──
def radial_power(X: np.ndarray, n_bins: int = 16) -> np.ndarray:
    """Mean power |FFT|² of a stack of images in rings of |k| (cycles/pixel), skipping the k = 0 ring.

    Images are normalised to zero mean, so the k = 0 (DC) ring carries no information and is left out.
    """
    P = np.abs(np.fft.fft2(X)) ** 2
    k = radial_frequency(X.shape[1:])
    b = np.minimum((k * 2 * n_bins).astype(int), n_bins - 1)
    return np.array([P[:, b == i].mean() for i in range(1, n_bins)])


fig = plt.figure(figsize=(10, 4.2))
for c in range(len(CAST)):
    for row, (X, y, lab) in enumerate([(Xs_test, ys_test, "source"), (Xt_test, yt_test, "target")]):
        ax = fig.add_subplot(2, 6, row * 6 + c + 1)
        ax.imshow(X[np.where(y == c)[0][0]])
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(NAMES[c], fontsize=8)
        if c == 0:
            ax.set_ylabel(lab, fontsize=9)
ax = fig.add_subplot(1, 3, 3)
freqs = (np.arange(1, 16) + 0.5) / 32           # ring centres in cycles/pixel (k = 0 ring skipped)
ax.semilogy(freqs, radial_power(Xs_test), "o-", label="source (simulator)")
ax.semilogy(freqs, radial_power(Xt_test), "s-", label="target ('real')")
ax.set_xlabel("spatial frequency (cycles/pixel)\n0.5 = Nyquist = 1/(20 Å)")
ax.set_ylabel("mean power (log scale)"); ax.legend(fontsize=8); ax.set_title("radially averaged power spectrum")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The same molecules in the same kind of random orientations — but the target images have
blurrier blobs, low-frequency "clouds" (coloured noise), partial neighbours at the edges and a different CTF ripple.
The power spectra make the difference *quantitative*: relative to the source, the target has more power at low
frequencies and less at high ones (all images are normalised to the same total variance). A human biologist would still recognise the ribosome; a network trained only on the left column has never
seen anything like the right column.

✏️ **Your turn.** Change one entry of `TARGET` (e.g. `noise_beta=0.0`, or `neighbour=0.0`) and re-run 1.3–1.4. Which
change makes the spectra most similar? (Later, re-run Section 2 too and see whether the accuracy drop shrinks.)

---
## 2 · The accuracy drop: train on source, test on target

We use a small 2D CNN (Module 05): four 3 × 3 convolution layers with max pooling, a global average pool that turns the
image into a 32-number **feature vector** $f(x)$, and a linear **classifier** $C(f)$ that outputs one score per molecule.
Keeping the feature extractor and the classifier as separate parts will matter for adaptation.

In [ ]:
# ── 2.1 Model, training and evaluation helpers ──
class SmallCNN(nn.Module):
    """Feature extractor f (conv → 32-D vector) followed by a linear classifier C."""

    def __init__(self, n_classes: int = 4, width: int = 8):
        super().__init__()
        w = width
        self.features = nn.Sequential(
            nn.Conv2d(1, w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(w, 2 * w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(2 * w, 4 * w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(4 * w, 4 * w, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.classifier = nn.Linear(4 * w, n_classes)

    def forward(self, x):
        return self.classifier(self.features(x))


def as_tensor(X: np.ndarray) -> torch.Tensor:
    """[n, H, W] NumPy images → [n, 1, H, W] float tensor on DEVICE."""
    return torch.tensor(X[:, None], dtype=torch.float32, device=DEVICE)


def train_classifier(X, y, steps=400, lr=2e-3, batch=64, seed=SEED, model=None):
    """Plain supervised training with cross-entropy (Module 05). Returns the trained model."""
    torch.manual_seed(seed)
    model = (model or SmallCNN()).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt_, yt_ = as_tensor(X), torch.tensor(y, device=DEVICE)
    for _ in range(steps):
        b = torch.randint(0, len(X), (batch,))
        loss = F.cross_entropy(model(Xt_[b]), yt_[b])
        opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()


@torch.no_grad()
def predict(model, X) -> np.ndarray:
    """Predicted class index for each image."""
    model.eval()
    return model(as_tensor(X)).argmax(1).cpu().numpy()


@torch.no_grad()
def features_of(model, X) -> np.ndarray:
    """The 32-D feature vector f(x) of each image."""
    model.eval()
    return model.features(as_tensor(X)).cpu().numpy()


def accuracy(model, X, y) -> float:
    """Fraction of correctly classified images."""
    return float((predict(model, X) == y).mean())


RESULTS = {}   # target accuracy of every method, for the summary in Section 8

In [ ]:
# ── 2.2 Train on the source only ──
t0 = time.time()
model_src = train_classifier(Xs, ys)
acc_src_src, acc_src_tgt = accuracy(model_src, Xs_test, ys_test), accuracy(model_src, Xt_test, yt_test)
RESULTS["source only"] = acc_src_tgt
print(f"trained in {time.time() - t0:.1f} s")
print(f"accuracy on SOURCE test: {acc_src_src:.3f}   |   on TARGET test: {acc_src_tgt:.3f}   (chance = 0.25)")
assert acc_src_src > acc_src_tgt + 0.15, "expected a clear accuracy drop on the target domain"

In [ ]:
# ── 2.3 Where does it go wrong? Confusion matrices ──
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6))
for ax, (X, y, title) in zip(axes, [(Xs_test, ys_test, "source test"), (Xt_test, yt_test, "target test")]):
    cm = confusion_matrix(y, predict(model_src, X), normalize="true")
    ax.imshow(cm, cmap="viridis", vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] < 0.6 else "black")
    ax.set_xticks(range(4)); ax.set_xticklabels([p.upper() for p in CAST], fontsize=8)
    ax.set_yticks(range(4)); ax.set_yticklabels([p.upper() for p in CAST], fontsize=8)
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"source-only model on {title}")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On source images the network is good; on target images the confusion matrix smears out and
whole classes are pulled into one or two "attractor" predictions (read the columns of the right panel). This is
typical of domain shift: the cues the network relied on (fine edges, the CTF fringe pattern, a clean background) look
different in the target, so its decisions fall back on whatever cue survives — not on the molecule's shape.

### 2.4 Seeing the gap in feature space

How different do the two domains look **to the network**? Two diagnostics:

1. **A picture:** project the 32-D features of source and target images onto their first two principal components (PCA).
2. **A number:** train a simple *domain classifier* (logistic regression) to tell source features from target features.
   If it succeeds with accuracy near 1, the domains are far apart; near 0.5, they are indistinguishable. The
   **proxy A-distance** $\hat d_A = 2\,(1 - 2\,\epsilon)$, with $\epsilon$ the domain classifier's error, turns this into a
   distance between 0 (same) and 2 (completely separate) — a practical stand-in for the middle term of the bound in 1.1.

In [ ]:
# ── 2.4 Feature-space picture and the proxy A-distance ──
def domain_gap(model, Xa, Xb, seed=SEED) -> float:
    """Proxy A-distance 2(1 − 2·err) of a logistic-regression domain classifier on the model's features."""
    Fa, Fb = features_of(model, Xa), features_of(model, Xb)
    Z = np.concatenate([Fa, Fb]); d = np.r_[np.zeros(len(Fa)), np.ones(len(Fb))]
    acc = cross_val_score(LogisticRegression(max_iter=2000), Z, d, cv=3).mean()
    return 2 * (1 - 2 * (1 - acc))


def plot_feature_space(model, title, ax_dom, ax_cls):
    """PCA of source-test and target-test features coloured by domain (left) and by true class (right)."""
    Fa, Fb = features_of(model, Xs_test), features_of(model, Xt_test)
    Z = PCA(2, random_state=SEED).fit_transform(np.concatenate([Fa, Fb]))
    ax_dom.scatter(*Z[:len(Fa)].T, s=4, alpha=0.5, label="source")
    ax_dom.scatter(*Z[len(Fa):].T, s=4, alpha=0.5, label="target")
    ax_dom.legend(fontsize=7, markerscale=3); ax_dom.set_title(f"{title}: by domain", fontsize=9)
    for c in range(4):
        sel = np.r_[ys_test, yt_test] == c
        ax_cls.scatter(*Z[sel].T, s=4, alpha=0.5, label=CAST[c].upper())
    ax_cls.legend(fontsize=7, markerscale=3); ax_cls.set_title(f"{title}: by true class", fontsize=9)
    for a in (ax_dom, ax_cls):
        a.set_xlabel("PC 1"); a.set_ylabel("PC 2")


fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6))
plot_feature_space(model_src, "source-only", *axes)
plt.tight_layout(); plt.show()
gap_src = domain_gap(model_src, Xs_test, Xt_test)
print(f"proxy A-distance between domains (source-only features): {gap_src:.2f}  (0 = identical, 2 = fully separable)")

🔍 **What just happened?** In the first two principal components the domains overlap only partly, and the domain
classifier, which uses all 32 feature dimensions, separates them well (a proxy A-distance well above 1). The class
clouds of the target are shifted relative to the source ones, so the classifier $C$, whose boundaries were drawn around
the *source* clouds, puts many target points on the wrong side. Two strategies follow:

- **make the source broader**, so the target cloud lies inside it (Sections 3 and 6), or
- **make the features domain-blind**, so the two clouds overlap (Sections 4–5).

---
## 3 · Domain randomization: train on many microscopes at once

🧫 If you do not know exactly what the real microscope does, simulate **many** microscopes: draw a new defocus, B-factor,
tilt range, SNR, noise colour and crowding level for **every** training image. The network can no longer rely on any
one imaging setting and is pushed to learn the *molecule*. If the real data look like "one more random draw", the
network should generalise. This idea comes from robotics (Tobin *et al.*, 2017: train in simulation with random
textures and lighting, deploy on a real robot).

Formally, instead of minimising the loss on one source distribution we minimise its average over a distribution of
simulator parameters $\phi$:

$$\min_\theta\; \mathbb E_{\phi\sim p(\phi)}\;\mathbb E_{(x,y)\sim P_S^{\phi}}\big[\mathcal L\big(C_\theta(x), y\big)\big].$$

*In words:* the network must do well on average over all the microscopes we can imagine, not just one.

In [ ]:
# ── 3.1 Randomized imaging parameters ──
def random_microscope(rng: np.random.Generator) -> dict:
    """One random draw of imaging parameters (ranges chosen to bracket plausible microscopes)."""
    return dict(defocus_um=rng.uniform(1.5, 6.0), B=rng.uniform(0, 300), tilt_max=rng.uniform(45, 70),
                snr=float(np.exp(rng.uniform(np.log(0.3), np.log(2.0)))), noise_beta=rng.uniform(0, 1.0),
                neighbour=rng.uniform(0, 0.6))


t0 = time.time()
X_rand = np.stack([microscope(clean[i], rng, pool=clean, **random_microscope(rng)) for i in idx_src_train]).astype(np.float32)
print(f"generated {len(X_rand)} randomized training images in {time.time() - t0:.1f} s")
fig, axes = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img in zip(axes, X_rand[:8]):
    ax.imshow(img); ax.axis("off")
fig.suptitle("eight randomized 'microscopes' (same simulator, random settings)", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ── 3.2 Train on randomized data ──
t0 = time.time()
model_rand = train_classifier(X_rand, ys)
RESULTS["randomization"] = accuracy(model_rand, Xt_test, yt_test)
print(f"trained in {time.time() - t0:.1f} s")
print(f"randomized model: source test {accuracy(model_rand, Xs_test, ys_test):.3f} | "
      f"target test {RESULTS['randomization']:.3f}  (source-only: {RESULTS['source only']:.3f})")

🔍 **What just happened?** Randomization usually costs a little accuracy on the original source (the task became harder)
and buys a lot on the target, because the target settings now fall *inside* the training distribution. It is not
magic: if the target needs something the randomizer never produces, randomization cannot help.

✏️ **Your turn.** In `random_microscope`, shrink the ranges so that they no longer cover the target (e.g.
`noise_beta=rng.uniform(0, 0.2)` and `neighbour=rng.uniform(0, 0.1)`), re-run 3.1–3.2 and compare. *Hint: the target used
`noise_beta=0.8, neighbour=0.5`.*

### 📄 Paper connection — Domain randomization (Che *et al.*, BIBM 2019)

- **Paper**: Che C, Xian Z, Zeng X, Gao X, Xu M. *Domain randomization for macromolecule structure classification and
  segmentation in electron cyro-tomograms* ("cyro" as spelled in the lab's entry). IEEE BIBM 2019, pp. 6–11.
- ⚠️ **We could not access the full text of this paper while writing; the description below is limited to the title
  and the released code. Read the paper before relying on details.**
- **Question it answers**: can models trained on simulated subtomograms with *randomized* simulation parameters transfer
  to real data, for both classification and segmentation?
- **What we know from the code**: AITom `aitom/classify/deep/supervised/cnn/domain_randomization/` contains only the
  segmentation part — a 2D/3D U-Net trained with cross-entropy and validated with the Dice coefficient. Which parameters
  the paper randomizes, their ranges, and its results are **not verified**.
- **What our toy keeps**: the general technique (random SNR, defocus, B-factor, wedge, noise colour and crowding per image).
- **For a verified example of randomization ranges**, see the Cryo-Shift box in Section 4 (SNR 0.03–10, wedge 0–50°,
  defocus and C_s) — do not attribute those ranges to this paper.

---
## 4 · Adversarial adaptation: make the features domain-blind

🧫 We *do* have target images — just no labels. Can unlabelled real images still teach the network something? Yes: they
tell us what the target **looks like**, and we can demand that the network's features look the same for source and target.

### 4.1 The min–max game

Add a third network, a **domain discriminator** $D$, that sees a feature vector and guesses "source (0) or target (1)?".
Then play a game:

- $D$ tries to **win**: minimise its domain-classification loss $\mathcal L_{dom}$ (binary cross-entropy).
- the feature extractor $f$ tries to make $D$ **lose**, while still classifying source images correctly:

$$\min_{f,\,C}\;\Big[\underbrace{\mathcal L_{cls}\big(C(f(x_s)), y_s\big)}_{\text{be right on source}}\;-\;\lambda\,
\underbrace{\mathcal L_{dom}\big(D(f(x)), d\big)}_{\text{fool the discriminator}}\Big],\qquad
\min_D\;\mathcal L_{dom}\big(D(f(x)), d\big).$$

*In words:* the features must carry enough information to classify molecules, but **no** information about which
microscope made the image. If $D$ cannot tell, the source and target clouds of Section 2.4 overlap, and the source
classifier works on both.

### 4.2 The gradient-reversal trick (DANN)

Ganin & Lempitsky (2015) made this game trainable with **one** ordinary backward pass. Insert a **gradient-reversal
layer** (GRL) between $f$ and $D$: in the forward pass it is the identity, in the backward pass it multiplies the
gradient by $-\lambda$:

$$\mathrm{GRL}_\lambda(z) = z, \qquad \frac{\partial\, \mathrm{GRL}_\lambda}{\partial z} = -\lambda I.$$

*In words:* $D$ receives normal gradients and gets better at telling domains apart; $f$ receives the *opposite* gradients
and gets better at hiding the domain. The weight $\lambda$ is ramped up from 0 to $\lambda_{max}$ with
$\lambda_p = \lambda_{max}\big(\tfrac{2}{1+e^{-10p}}-1\big)$, $p$ = training progress, so that the classifier first learns
something useful. This network is called **DANN** (domain-adversarial neural network).

In [ ]:
# ── 4.2 The gradient-reversal layer and DANN training ──
class GradReverse(torch.autograd.Function):
    """Identity in the forward pass; multiplies the gradient by −λ in the backward pass."""

    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None


def adapt_dann(model, X_src, y_src, X_tgt, steps=300, lam_max=1.0, lr=1e-3, batch=64, seed=SEED,
               monitor=None):
    """DANN fine-tuning: source cross-entropy + domain-discriminator loss through a gradient-reversal layer.

    ``monitor=(X, y)`` records accuracy on a labelled set every 50 steps (for teaching only — in real life you
    have no target labels to monitor!). Returns (adapted model, history).
    """
    torch.manual_seed(seed)
    model = copy.deepcopy(model).to(DEVICE).train()
    width = model.classifier.in_features
    disc = nn.Sequential(nn.Linear(width, 64), nn.ReLU(), nn.Linear(64, 1)).to(DEVICE)
    opt = torch.optim.Adam(list(model.parameters()) + list(disc.parameters()), lr=lr)
    Xs_, ys_, Xt_ = as_tensor(X_src), torch.tensor(y_src, device=DEVICE), as_tensor(X_tgt)
    dom = torch.cat([torch.zeros(batch), torch.ones(batch)]).to(DEVICE)       # 0 = source, 1 = target
    history = []
    for step in range(steps):
        lam = lam_max * (2 / (1 + np.exp(-10 * step / steps)) - 1)
        bs, bt = torch.randint(0, len(X_src), (batch,)), torch.randint(0, len(X_tgt), (batch,))
        f_s, f_t = model.features(Xs_[bs]), model.features(Xt_[bt])
        loss_cls = F.cross_entropy(model.classifier(f_s), ys_[bs])
        d_logit = disc(GradReverse.apply(torch.cat([f_s, f_t]), lam)).squeeze(1)
        loss_dom = F.binary_cross_entropy_with_logits(d_logit, dom)
        opt.zero_grad(); (loss_cls + loss_dom).backward(); opt.step()
        if monitor is not None and (step + 1) % 50 == 0:
            history.append((step + 1, accuracy(model, *monitor), loss_dom.item()))
            model.train()
    return model.eval(), history


t0 = time.time()
model_dann, hist_dann = adapt_dann(model_src, Xs, ys, Xt, monitor=(Xt_test, yt_test))
RESULTS["DANN (adversarial)"] = accuracy(model_dann, Xt_test, yt_test)
print(f"DANN fine-tuning in {time.time() - t0:.1f} s")
print(f"DANN model: source test {accuracy(model_dann, Xs_test, ys_test):.3f} | target test {RESULTS['DANN (adversarial)']:.3f}")
for step, acc, ld in hist_dann:
    print(f"  step {step:3d}: target accuracy {acc:.3f}, discriminator loss {ld:.3f}  (ln 2 = 0.693 means 'cannot tell')")

In [ ]:
# ── 4.3 Did the clouds move together? ──
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.6))
plot_feature_space(model_dann, "after DANN", *axes)
plt.tight_layout(); plt.show()
gap_dann = domain_gap(model_dann, Xs_test, Xt_test)
print(f"proxy A-distance: source-only {gap_src:.2f} → after DANN {gap_dann:.2f}")

🔍 **What just happened?** We never showed the network a single target label, yet target accuracy rose above the
source-only value, and the discriminator's loss moved towards ln 2 ≈ 0.69 — the value of a coin flip. Look at the
proxy A-distance, though: a *fresh* domain probe can still tell the domains apart. Fooling one discriminator during
training is not the same as making the domains indistinguishable to every classifier; DANN helps because it removes
some of the domain cues that misled the class boundaries, not because it erases the domain completely. Three warnings
that every practitioner learns the hard way:

- ⚠️ **Adversarial training is unstable.** The target-accuracy curve wobbles from step to step; too large a $\lambda$
  can wreck the source accuracy.
- ⚠️ **"Domain-blind" is a goal, not a guarantee** — check it with an independent probe, as we just did.
- ⚠️ **We cheated by monitoring the target labels** in the printout. In real life you cannot; you must choose
  $\lambda$ and the stopping step without them (e.g. with the discriminator loss or a small labelled validation set).

Aligning the *marginal* feature distributions can also align the **wrong** classes with each other (e.g. target
proteasomes onto source RuBisCO) if class proportions differ between domains. Cryo-Shift's class-wise discriminators
(below) address exactly this.

### 📄 Paper connection — 3D-ADA (Lin, Zeng *et al.*, ISMB 2019)

- **Paper**: Lin R, Zeng X, Kitani K, Xu M (Lin and Zeng joint first authors). *Adversarial domain adaptation for cross
  data source macromolecule in situ structural classification in cellular electron cryo-tomograms.* Bioinformatics
  35(14):i260–i268 (ISMB 2019). [doi:10.1093/bioinformatics/btz364](https://doi.org/10.1093/bioinformatics/btz364).
  (The lab's BibTeX misspells the second author as "Zheng"; it is Zeng.)
- **Question it answers**: a subtomogram classifier trained on one data source (simulated data, or another microscope)
  fails on another; target labels are expensive. Can unlabelled target subtomograms fix it?
- **What the paper actually does**: an **ADDA-style, two-stage** scheme (Tzeng *et al.* 2017) rather than DANN:
  (1) train a source extractor $F_s$ and classifier $C$ on source labels; (2) freeze them, initialise a **separate**
  target extractor $F_t \leftarrow F_s$ and train it against a discriminator $D$ (source label 0, target label 1):
  $$\min_D -\mathbb E_{x\sim X_t}[\log D(F_t(x))] - \mathbb E_{x\sim X_s}[\log(1-D(F_s(x)))],\qquad
    \min_{F_t} -\mathbb E_{x\sim X_t}[\log(1-D(F_t(x)))].$$
  The target extractor is trained to make target features look like *source* features (an "inverted label" loss,
  which the authors describe as avoiding vanishing gradients); predictions are $C(F_t(x_t))$.
- **What our toy keeps**: unlabelled target images + a feature-level domain discriminator + an adversarial objective.
- **What our toy leaves out**: separate source/target extractors and the two-stage ADDA schedule (we use DANN's single
  network with gradient reversal), 3D subtomograms, and the paper's data (23 simulated classes at 40³ voxels and two
  imaging settings; three experimental datasets).
- **Result reported**: the authors report that 3D-ADA was best in 24 of 25 simulated SNR combinations (e.g. S_B→S_A at
  SNR 1000: 0.855 → 0.991 accuracy) and improved all experimental cases (e.g. 0.313 → 0.688 on one dataset).
- **Code**: AITom `aitom/classify/deep/supervised/cnn/domain_adaptation_adversarial/` (Chainer; `updater.py` holds the
  ADDA update).

### 4.4 Cryo-Shift-lite: randomization **and** adversarial adaptation

Nothing stops us from combining Sections 3 and 4: train on randomized source images *and* align features with the
unlabelled target. This is the recipe of **Cryo-Shift**.

In [ ]:
# ── 4.4 Randomization + DANN ──
t0 = time.time()
model_cs, hist_cs = adapt_dann(model_rand, X_rand, ys, Xt, monitor=(Xt_test, yt_test))
RESULTS["randomization + DANN"] = accuracy(model_cs, Xt_test, yt_test)
print(f"trained in {time.time() - t0:.1f} s")
print(f"randomization + DANN: target test {RESULTS['randomization + DANN']:.3f} "
      f"(randomization alone {RESULTS['randomization']:.3f}, DANN alone {RESULTS['DANN (adversarial)']:.3f})")

### 📄 Paper connection — Cryo-Shift (Bandyopadhyay *et al.*, Bioinformatics 2022)

- **Paper**: Bandyopadhyay H, Deng Z, Ding L, Liu S, Uddin MR, Zeng X, Behpour S, Xu M. *Cryo-shift: reducing domain
  shift in cryo-electron subtomograms with unsupervised domain adaptation and randomization.* Bioinformatics
  38(4):977–984 (2022; online Nov 2021). [doi:10.1093/bioinformatics/btab794](https://doi.org/10.1093/bioinformatics/btab794),
  [arXiv:2111.09114](https://arxiv.org/abs/2111.09114).
- **Question it answers**: adapt a simulated-data classifier to real subtomograms with **zero** real labels.
- **What the paper actually does**: (a) **network-driven randomization** (DeceptionNet-style): a warp network $W$ with a
  bounded learned-noise module and a learned dense-deformation module perturbs simulated inputs, trained adversarially
  through a gradient-reversal layer; the simulator itself also randomizes SNR (0.03–10, log scale), maximum wedge angle
  (0–50°), defocus and C_s. (b) **unsupervised domain adaptation** with an image-level domain discriminator $D$ and **one
  feature discriminator per class** $F_i$, each weighted by the classifier's predicted class probability, trained through
  gradient reversal:
  $\mathcal L_{train} = \mathcal L_{task}\big(M(W(S^{(h)}(d_{map}))), y\big) + \mathcal L_{dd}(D, t_d) + \mathcal L_{fd}(F, t_f)$.
  No pseudo-labelling or self-training is used.
- **What our toy keeps**: randomized simulation + gradient-reversal feature alignment with unlabelled target images.
- **What our toy leaves out**: the learned warp network, the image-level discriminator, class-wise feature
  discriminators, and 3D data (4 simulated classes at 40³ voxels; 1,051 real rat-neuron subtomograms).
- **Result reported**: the authors report that warp randomization raised CB3D macro accuracy from 0.71 to 0.77, and the
  full method reached 0.90 accuracy and 0.86 macro-F1 on held-out real subtomograms, above few-shot methods that do use
  real labels (the paper's tables use different summary metrics — quote one table consistently).
- **Code**: AITom `aitom/classify/deep/unsupervised/cryo_shift/`.

---
## 5 · When you can afford a few labels: few-shot domain adaptation

🧫 Sometimes an expert *can* label a handful of real particles — say 3 per class. Two simple tools:

1. **Fine-tune** the source model on those few labelled target images (risk: over-fitting to 12 images).
2. **Deep CORAL** (Sun & Saenko, 2016): align the *second-order statistics* (covariances) of source and target features
   using all the unlabelled target images,

$$\mathcal L_{CORAL} = \frac{1}{4d^2}\,\big\lVert C^S - C^T\big\rVert_F^2,$$

where $C^S$ and $C^T$ are the $d\times d$ covariance matrices of a batch of source and target features and
$\lVert\cdot\rVert_F$ is the Frobenius norm (square root of the sum of squared entries). *In words:* the spread and
correlations of the features must look the same in both domains — a gentler, non-adversarial alignment.

In [ ]:
# ── 5.1 Few-shot fine-tuning, with and without Deep CORAL ──
def coral_loss(fs: torch.Tensor, ft: torch.Tensor) -> torch.Tensor:
    """Deep CORAL: squared Frobenius distance between source and target feature covariances / (4 d²)."""
    d = fs.shape[1]
    cs, ct_ = torch.cov(fs.T), torch.cov(ft.T)
    return ((cs - ct_) ** 2).sum() / (4 * d * d)


def adapt_few_shot(model, X_src, y_src, X_few, y_few, X_unlab, steps=150, coral_weight=0.0, lr=1e-3, seed=SEED):
    """Fine-tune on source + a few labelled target images; optionally add CORAL on unlabelled target features."""
    torch.manual_seed(seed)
    model = copy.deepcopy(model).to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xs_, ys_ = as_tensor(X_src), torch.tensor(y_src, device=DEVICE)
    Xf_, yf_ = as_tensor(X_few), torch.tensor(y_few, device=DEVICE)
    Xu_ = as_tensor(X_unlab)
    for _ in range(steps):
        bs, bu = torch.randint(0, len(X_src), (64,)), torch.randint(0, len(X_unlab), (64,))
        f_s = model.features(Xs_[bs])
        loss = F.cross_entropy(model.classifier(f_s), ys_[bs]) + F.cross_entropy(model(Xf_), yf_)
        if coral_weight > 0:
            loss = loss + coral_weight * coral_loss(f_s, model.features(Xu_[bu]))
        opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()


K_SHOT = 3
few_idx = np.concatenate([np.where(yt_hidden == c)[0][:K_SHOT] for c in range(4)])   # an expert labels 3 per class
X_few, y_few = Xt[few_idx], yt_hidden[few_idx]
t0 = time.time()
model_fs = adapt_few_shot(model_src, Xs, ys, X_few, y_few, Xt)
model_fs_coral = adapt_few_shot(model_src, Xs, ys, X_few, y_few, Xt, coral_weight=50.0)
RESULTS[f"{K_SHOT}-shot fine-tune"] = accuracy(model_fs, Xt_test, yt_test)
RESULTS[f"{K_SHOT}-shot + CORAL"] = accuracy(model_fs_coral, Xt_test, yt_test)
print(f"trained both in {time.time() - t0:.1f} s")
print(f"{K_SHOT} labelled target images per class → target test {RESULTS[f'{K_SHOT}-shot fine-tune']:.3f}; "
      f"+ CORAL on unlabelled target → {RESULTS[f'{K_SHOT}-shot + CORAL']:.3f}")

🔍 **What just happened?** Twelve labelled target images are already informative, and CORAL lets the thousands of
*unlabelled* target images contribute too. Results with so few labels depend on *which* 3 images per class you pick —
re-run with other indices (✏️ change `[:K_SHOT]` to `[5:5 + K_SHOT]`) and watch the numbers move.

### 📄 Paper connection — Few-shot domain adaptation (Yu *et al.*, Bioinformatics 2021)

- **Paper**: Yu L, Li R, Zeng X, Wang H, Jin J, Yang G, Jiang R, Xu M. *Few shot domain adaptation for in situ
  macromolecule structural classification in cryoelectron tomograms.* Bioinformatics 37(2):185–191 (2021; online July
  2020). [doi:10.1093/bioinformatics/btaa671](https://doi.org/10.1093/bioinformatics/btaa671),
  [arXiv:2007.15422](https://arxiv.org/abs/2007.15422) (the preprint lists the authors in a different order).
- **Question it answers**: plenty of labelled (often simulated) source data, but only 3–7 labels per class in the real target.
- **What the paper actually does**: Stage 1 — source training with cross-entropy. Stage 2.1 — add the **Deep CORAL**
  loss on unlabelled target features ($L^{CORAL} + L^C$). Stage 2.2 — train a **domain-class discriminator** *once*,
  with the encoder fixed, that classifies feature *pairs* into 4 groups (same/different domain × same/different class).
  Stage 3 — freeze it and fine-tune encoder and classifier with $L=\gamma L^{DCD}+L^s+L^t$, where $L^{DCD}$ pushes
  source–target pairs to look like same-domain pairs. Unlike FADA, there is **no alternating adversarial training**
  (an alternating variant did slightly worse, 94.2 % vs 95.3 %).
- **What our toy keeps**: few labelled target samples + Deep CORAL on unlabelled target features.
- **What our toy leaves out**: the 4-group pair discriminator and Stage 3, domain-specific early convolution blocks, 3D data.
- **Result reported**: the authors report accuracy after each stage of 37.2 % → 70.9 % (CORAL) → 70.9 % → 95.3 %
  (discriminator fine-tuning), and 79.1 % without CORAL; they beat CORAL, sliced-Wasserstein, fine-tuning and FADA in
  most settings (simulated: 43 classes; real: rat neuron tomograms and the Noble single-particle subtomograms).
- **Code**: AITom `aitom/classify/deep/supervised/cnn/few_shot/domain_adaptation/` (PyTorch, `CORAL.py`).

---
## 6 · Learning the microscope from unpaired data (CryoETGAN)

🧫 Instead of changing the *network*, change the *training data*: learn a mapping $G$ that turns clean simulated
densities into images that look like the real microscope's, then train any classifier on $G$'s output with the free
simulation labels. The catch: we have **no pairs** (we never see the same molecule both clean and real).

### 6.1 Unpaired translation: GANs and cycle consistency

A **generative adversarial network** (GAN) trains $G$ against a **critic** (discriminator) $D_Y$ that tries to tell real
target images $y$ from generated ones $G(x)$. A **CycleGAN** (Zhu *et al.*, 2017) adds a second generator $F$ for the
way back and demands that a round trip returns the input:

$$\mathcal L = \mathcal L_{GAN}(G, D_Y) + \mathcal L_{GAN}(F, D_X) + \lambda\Big(\mathbb E\lVert F(G(x))-x\rVert_1
  + \mathbb E\lVert G(F(y))-y\rVert_1\Big).$$

*In words:* $G(x)$ must look real (fool the critic) **and** still contain enough of $x$ to reconstruct it — so $G$ cannot
simply output any random real-looking image (that would break the cycle).

A **Wasserstein GAN** (WGAN) replaces the critic's "real or fake?" probability by an unbounded score and trains it to
maximise $\mathbb E[D(y)] - \mathbb E[D(G(x))]$ — an estimate of the Wasserstein ("earth-mover's") distance between the
two image distributions — while keeping $D$ smooth (1-Lipschitz) with weight clipping or **spectral normalisation**.

### 6.2 A tiny physics-constrained "learned microscope"

Training a full CycleGAN takes hours. Our toy keeps the *idea* (learn the imaging process from **unpaired** real images
with a Wasserstein critic) but gives $G$ only a few physically meaningful knobs, so it cannot invent molecules and it
trains in seconds:

$$G(x) = \mathrm{normalize}\big(h * x \;+\; \sigma\,(g * n)\big),\qquad n\sim\mathcal N(0, I),$$

with a learnable 9 × 9 point-spread kernel $h$ (playing CTF + blur + wedge), a learnable 7 × 7 noise-colouring kernel $g$
and a learnable noise level $\sigma$. Because $h$ is a convolution, $G$ keeps the molecule in place — the job the cycle
loss does in a full CycleGAN.

In [ ]:
# ── 6.2 Generator, critic and adversarial training ──
class LearnedMicroscope(nn.Module):
    """G(x) = normalize(h * x + σ·(g * noise)) with learnable kernels h, g and noise level σ."""

    def __init__(self, k_psf: int = 9, k_noise: int = 7):
        super().__init__()
        h = torch.zeros(1, 1, k_psf, k_psf); h[0, 0, k_psf // 2, k_psf // 2] = 1.0     # start as "do nothing"
        g = torch.zeros(1, 1, k_noise, k_noise); g[0, 0, k_noise // 2, k_noise // 2] = 1.0
        self.h, self.g = nn.Parameter(h), nn.Parameter(g)
        self.log_sigma = nn.Parameter(torch.tensor(0.0))

    def forward(self, x):
        signal = F.conv2d(x, self.h, padding=self.h.shape[-1] // 2)
        noise = F.conv2d(torch.randn_like(x), self.g, padding=self.g.shape[-1] // 2)
        y = signal + self.log_sigma.exp() * noise
        y = y - y.mean((2, 3), keepdim=True)
        return y / (y.std((2, 3), keepdim=True) + 1e-6)


def make_critic() -> nn.Module:
    """A small CNN critic with spectral normalisation (keeps it ~1-Lipschitz, as in CryoETGAN's discriminators)."""
    SN = nn.utils.parametrizations.spectral_norm
    return nn.Sequential(
        SN(nn.Conv2d(1, 16, 3, padding=1)), nn.LeakyReLU(0.2), nn.AvgPool2d(2),
        SN(nn.Conv2d(16, 32, 3, padding=1)), nn.LeakyReLU(0.2), nn.AvgPool2d(2),
        SN(nn.Conv2d(32, 32, 3, padding=1)), nn.LeakyReLU(0.2), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        SN(nn.Linear(32, 1)))


def train_wgan(clean_imgs, real_imgs, steps=300, n_critic=3, lr_g=5e-3, lr_d=2e-4, seed=SEED):
    """Unpaired WGAN: the critic separates real target images from G(clean); G learns to fool it (RMSProp)."""
    torch.manual_seed(seed)
    G, D = LearnedMicroscope().to(DEVICE), make_critic().to(DEVICE)
    opt_g, opt_d = torch.optim.RMSprop(G.parameters(), lr=lr_g), torch.optim.RMSprop(D.parameters(), lr=lr_d)
    Xc, Xr = as_tensor(clean_imgs), as_tensor(real_imgs)
    w_hist = []
    for step in range(steps):
        for _ in range(n_critic):
            bc, br = torch.randint(0, len(Xc), (64,)), torch.randint(0, len(Xr), (64,))
            with torch.no_grad():
                fake = G(Xc[bc])
            loss_d = D(fake).mean() - D(Xr[br]).mean()        # critic maximises E[D(real)] − E[D(fake)]
            opt_d.zero_grad(); loss_d.backward(); opt_d.step()
        bc = torch.randint(0, len(Xc), (64,))
        loss_g = -D(G(Xc[bc])).mean()
        opt_g.zero_grad(); loss_g.backward(); opt_g.step()
        w_hist.append(-loss_d.item())
    return G.eval(), w_hist


clean_src = np.stack([normalize(clean[i]) for i in idx_src_train])     # labelled clean projections (source)
t0 = time.time()
G_learned, w_hist = train_wgan(clean_src, Xt)
print(f"WGAN trained in {time.time() - t0:.1f} s; learned noise level σ = {G_learned.log_sigma.exp().item():.2f}")
with torch.no_grad():
    X_gan = torch.cat([G_learned(as_tensor(clean_src[i:i + 256])) for i in range(0, len(clean_src), 256)]).cpu().numpy()[:, 0]

In [ ]:
# ── 6.3 Does the learned microscope look like the target? ──
fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].plot(np.convolve(w_hist, np.ones(10) / 10, mode="valid"))
axes[0].set_xlabel("generator step"); axes[0].set_ylabel("critic's Wasserstein estimate")
axes[0].set_title("distance real ↔ generated (smoothed)")
axes[1].semilogy(freqs, radial_power(Xs_test), "o-", label="source simulator")
axes[1].semilogy(freqs, radial_power(Xt_test), "s-", label="target")
axes[1].semilogy(freqs, radial_power(X_gan), "^-", label="learned microscope G")
axes[1].set_xlabel("cycles/pixel"); axes[1].set_ylabel("mean power"); axes[1].legend(fontsize=7)
axes[1].set_title("power spectra")
axes[2].imshow(np.concatenate([np.concatenate([X_gan[np.where(ys == c)[0][0]] for c in range(4)], 1),
                               np.concatenate([Xt_test[np.where(yt_test == c)[0][0]] for c in range(4)], 1)], 0))
axes[2].set_title("top: G(clean), bottom: real target", fontsize=9); axes[2].axis("off")
plt.tight_layout(); plt.show()

model_gan = train_classifier(X_gan, ys)
RESULTS["learned microscope (WGAN)"] = accuracy(model_gan, Xt_test, yt_test)
print(f"classifier trained on G(clean) images: target test {RESULTS['learned microscope (WGAN)']:.3f} "
      f"(source only {RESULTS['source only']:.3f})")

🔍 **What just happened?** Without ever seeing a matched pair, the critic pushed $G$ to reproduce the target's spectrum:
its power curve moves from the source's towards the target's. A classifier trained only on $G$'s images (with free
simulation labels) transfers better than one trained on the original simulator. What $G$ **cannot** learn here is
anything outside its few knobs — e.g. crowding neighbours — which is why full CycleGANs use deep generators (and why
they need tricks against *mode collapse* and hallucination).

### 📄 Paper connection — CryoETGAN (Wu *et al.*, Front. Physiol. 2022)

- **Paper**: Wu X, Li C, Zeng X, Wei H, Deng HW, Zhang J, Xu M. *CryoETGAN: Cryo-electron tomography image synthesis via
  unpaired image translation.* Front. Physiol. 13:760404 (2022).
  [doi:10.3389/fphys.2022.760404](https://doi.org/10.3389/fphys.2022.760404).
- **Question it answers**: physics-based simulators make subtomograms that do not look real; learn the mapping from clean
  density maps to realistic subtomograms **without paired data**.
- **What the paper actually does**: a **3D CycleGAN with a Wasserstein loss**. Generators (c7s1-32, d64, d128, six residual
  blocks, u64, u32, c7s1-1) receive **noise concatenated after downsampling** to fight mode collapse; discriminators
  (C64-C128-C256, 4³ convolutions) use **spectral normalisation**; training uses weight clipping and RMSProp. The released
  code uses an L1 cycle loss $\mathcal L_{cyc}=\lambda_1\mathbb E\lVert F(G(x))-x\rVert_1+\lambda_2\mathbb E\lVert G(F(y))-y\rVert_1$
  with $\lambda_1=\lambda_2=10$ (the factor 10 appears twice in `model.py`). No identity loss is described.
- **What our toy keeps**: unpaired real target images, a Wasserstein critic with spectral normalisation, RMSProp, and
  "train a classifier on generated data" as the evaluation.
- **What our toy leaves out**: the second generator and the cycle loss (our convolutional $G$ cannot move the molecule,
  which plays the cycle's role), deep generators, noise injection, 3D volumes.
- **Result reported**: on their real dataset S_e1 (1,600 subtomograms of proteasome, ribosome, TRiC — PDB 5MPA, 5T2C,
  4V94 — and membrane) the authors report CB3D classification accuracy of 76.4 % with CryoETGAN samples vs 19.7 % with
  traditional simulation (67.3 % vs 28.9 % on the Noble dataset); removing spectral normalisation dropped accuracy to
  26.9 %, and removing the Wasserstein loss caused mode collapse (57.1 %). Check the exact train/test protocol in the
  paper before quoting these numbers.
- **Code**: AITom `aitom/simulation/subtomogram/deep/cryoetgan/` (TensorFlow).

### 📄 Paper connection — Regularized adversarial training, RAT (Wu *et al.*, BIBM 2019)

- **Paper**: Wu X, Mao Y, Wang H, Zeng X, Gao X, Xing E, Xu M. *Regularized adversarial training (RAT) for robust
  cellular electron cryo tomograms classification.* IEEE BIBM 2019.
- ⚠️ **We could not access the full text of this paper while writing; the description below is limited to the title.
  Read the paper before relying on details.**
- **Question it answers (from the title)**: make subtomogram classifiers robust to adversarially perturbed inputs —
  another face of "test images differ from training images" — by adversarial training plus a regulariser. The
  regulariser, attack type, data and numbers are **unverified**. (Do not confuse *adversarial training*, which feeds the
  network worst-case perturbed inputs, with the *adversarial domain adaptation* of Section 4.)

---
## 7 · Segmentation across domains: Vox-UDA-lite

🧫 Classification gives one label per subtomogram; **segmentation** gives one label per voxel ("which voxels belong to the
molecule?", Module 16). Voxel-wise labels for real tomograms are even rarer than class labels, so simulated training
data are attractive — and domain shift bites again. **Vox-UDA** (AAAI 2025) noticed that the dominant difference between
simulated and real subtomograms is the **noise**, and built two cryo-ET-specific tools around that observation:

1. a **Noise Generation Module (NGM)** that *measures* the target's noise and injects the same amount into source images;
2. **Denoised Pseudo-Labelling (DPL)**: a *mean teacher* labels real images after a bilateral filter has removed some
   noise, and only confident labels are used to train the student.

Our 2D toy: segment the molecule (mask = clean projection above 30 % of its maximum) in projection images. Source
images are almost clean (SNR 20) and labelled; target images are very noisy (SNR 0.1) and unlabelled.

In [ ]:
# ── 7.1 Source and target segmentation data ──
def seg_domain(indices, snr, rng):
    """Projections + Gaussian noise at a given SNR → (images, masks). Mask = clean projection > 30 % of its max."""
    P = clean[indices] / clean[indices].reshape(len(indices), -1).max(1)[:, None, None]
    masks = (P > 0.3).astype(np.float32)
    X = np.stack([normalize(normalize(p) + rng.normal(0, np.sqrt(1 / snr), p.shape)) for p in P]).astype(np.float32)
    return X, masks


SNR_SOURCE, SNR_TARGET = 20.0, 0.1
Xs_seg, Ms_seg = seg_domain(idx_src_train[:800], SNR_SOURCE, rng)          # labelled
Xt_seg, Mt_hidden = seg_domain(idx_tgt_train[:800], SNR_TARGET, rng)       # unlabelled (masks hidden)
Xt_seg_test, Mt_seg_test = seg_domain(idx_tgt_test, SNR_TARGET, rng)        # answer key for evaluation
fig, axes = plt.subplots(1, 4, figsize=(7.5, 2.2))
for ax, img, t in zip(axes, [Xs_seg[0], Ms_seg[0], Xt_seg_test[0], Mt_seg_test[0]],
                      ["source image (SNR 20)", "source mask", "target image (SNR 0.1)", "target mask (hidden)"]):
    ax.imshow(img); ax.set_title(t, fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

### 7.2 A small segmentation network and the Dice score

Vox-UDA uses a **VoxResNet** backbone (a 3D residual network). Our 2D stand-in keeps the two ingredients that matter
here — residual connections and a large receptive field — with dilated 3 × 3 convolutions at full resolution, so no
decoder is needed. Quality is measured with the **Dice coefficient** (Module 16),

$$\mathrm{Dice} = \frac{2\,|P\cap G|}{|P| + |G|},$$

where $P$ is the set of predicted molecule pixels and $G$ the true ones; 1 = perfect overlap, 0 = none.

In [ ]:
# ── 7.2 Segmentation network, Dice and supervised training ──
class TinyVoxResNet2D(nn.Module):
    """Residual blocks of dilated 3×3 convolutions (dilation 1, 2, 4) → one logit per pixel."""

    def __init__(self, width: int = 12):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(1, w, 3, padding=1), nn.ReLU())
        self.block2 = nn.Sequential(nn.Conv2d(w, w, 3, padding=2, dilation=2), nn.ReLU(),
                                    nn.Conv2d(w, w, 3, padding=2, dilation=2))
        self.block3 = nn.Sequential(nn.Conv2d(w, w, 3, padding=4, dilation=4), nn.ReLU(),
                                    nn.Conv2d(w, w, 3, padding=4, dilation=4))
        self.head = nn.Conv2d(w, 1, 1)

    def forward(self, x, return_features=False):
        f1 = self.stem(x)
        f2 = F.relu(f1 + self.block2(f1))
        f3 = F.relu(f2 + self.block3(f2))
        out = self.head(f3)
        return (out, [f1, f2, f3]) if return_features else out


@torch.no_grad()
def dice(model, X, M) -> float:
    """Dice coefficient of thresholded predictions (p > 0.5) against masks M."""
    model.eval()
    pred = (torch.sigmoid(model(as_tensor(X))) > 0.5).float().cpu().numpy()[:, 0]
    return float(2 * (pred * M).sum() / (pred.sum() + M.sum() + 1e-9))


def train_segmenter(X, M, steps=250, lr=3e-3, noise_sigma=0.0, consistency=False, seed=SEED):
    """Supervised segmentation on source images; optionally inject noise (NGM) + feature-consistency loss."""
    torch.manual_seed(seed)
    model = TinyVoxResNet2D().to(DEVICE).train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X_, M_ = as_tensor(X), as_tensor(M)
    for _ in range(steps):
        b = torch.randint(0, len(X), (32,))
        x, m = X_[b], M_[b]
        if noise_sigma > 0:
            x_noisy = x + noise_sigma * torch.randn_like(x)                        # x^{s'} = x^s + ε
            out_n, feats_n = model(x_noisy, return_features=True)
            loss = F.binary_cross_entropy_with_logits(out_n, m)
            if consistency:                                                        # L_con: clean and noisy features agree
                out_c, feats_c = model(x, return_features=True)
                loss = loss + F.binary_cross_entropy_with_logits(out_c, m)
                loss = loss + sum(w * (1 - F.cosine_similarity(a.flatten(1), c.flatten(1)).mean())
                                  for w, a, c in zip([0.3, 0.3, 0.4], feats_n, feats_c))
        else:
            loss = F.binary_cross_entropy_with_logits(model(x), m)
        opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()


SEG = {}
t0 = time.time()
seg_src = train_segmenter(Xs_seg, Ms_seg)
SEG["1 · source only"] = dice(seg_src, Xt_seg_test, Mt_seg_test)
print(f"trained in {time.time() - t0:.1f} s | Dice on source: {dice(seg_src, Xs_seg[:200], Ms_seg[:200]):.3f} | "
      f"Dice on TARGET: {SEG['1 · source only']:.3f}")

### 7.3 Step 2 — the Noise Generation Module (NGM)

How noisy is the target, if we have no clean target images? Vox-UDA's answer: look at **high frequencies**, where a
molecule at this resolution has almost no signal, so what remains is mostly noise. Keep a fraction $\rho = 24.4\,\%$ of
the Fourier coefficients (the highest frequencies), transform back, and measure the variance:

$$\hat x'_n = H_{high}\odot\mathcal F[x^t_n],\qquad x^{t'}_n = \mathcal F^{-1}[\hat x'_n],\qquad
  \sigma_t^2 \approx \frac{1}{\rho}\cdot\frac{1}{N}\sum_{n=1}^{N}\mathrm{Var}\big(x^{t'}_n\big),\quad N = 10 .$$

Then add $\epsilon\sim\mathcal N(0,\sigma_t^2 I)$ to every source image: $x^{s'}_i = x^s_i + \epsilon$.

⚠️ **Our reading.** The paper estimates $\sigma_t^2$ from the variance of high-passed target samples; the factor
$1/\rho$ is **our** addition (for white noise, Parseval's theorem says a mask keeping a fraction $\rho$ of the
coefficients keeps a fraction $\rho$ of the noise variance). We also subtract the source's own small noise estimate.
The paper adds a **feature-consistency loss** $\mathcal L_{con}=\sum_k\lambda_k\,\mathcal L_{BN}(f_k, f'_k)$ (cosine
similarity between clean and noised features at several layers, $[\lambda]=[0.2,0.2,0.3,0.3]$ over four VoxResNet
layers); we use three layers with weights $[0.3, 0.3, 0.4]$.

In [ ]:
# ── 7.3 Estimate the target noise from high frequencies and train with injected noise ──
def estimate_noise_variance(X, rho=0.244, n_samples=10):
    """NGM noise estimate: variance of the top-ρ-frequency part of the images, divided by ρ (our Parseval correction)."""
    k = radial_frequency(X.shape[1:])
    H_high = (k >= np.quantile(k, 1 - rho)).astype(np.float32)
    v = np.mean([np.real(np.fft.ifft2(np.fft.fft2(x) * H_high)).var() for x in X[:n_samples]])
    return float(v / H_high.mean()), H_high


var_t, H_high = estimate_noise_variance(Xt_seg)
var_s, _ = estimate_noise_variance(Xs_seg)
sigma_t = np.sqrt(max(var_t - var_s, 0.0))
true_var = 1 / (1 + SNR_TARGET)    # noise variance of a normalised image with this SNR (we know it only in simulation)
print(f"estimated target noise variance {var_t:.3f} (true {true_var:.3f}); source {var_s:.3f} → inject σ = {sigma_t:.2f}")

fig, axes = plt.subplots(1, 3, figsize=(7.5, 2.4))
axes[0].imshow(np.fft.fftshift(H_high)); axes[0].set_title(f"high-pass mask (ρ = {H_high.mean():.3f})", fontsize=8)
axes[1].imshow(np.real(np.fft.ifft2(np.fft.fft2(Xt_seg[0]) * H_high))); axes[1].set_title("high-passed target ≈ noise", fontsize=8)
axes[2].imshow(Xs_seg[0] + sigma_t * np.random.default_rng(1).normal(size=Xs_seg[0].shape))
axes[2].set_title("source + generated noise", fontsize=8)
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

t0 = time.time()
seg_ngm = train_segmenter(Xs_seg, Ms_seg, steps=200, noise_sigma=sigma_t, consistency=True)
SEG["2 · + noise generation"] = dice(seg_ngm, Xt_seg_test, Mt_seg_test)
print(f"trained in {time.time() - t0:.1f} s | Dice on TARGET: {SEG['2 · + noise generation']:.3f} "
      f"(source only {SEG['1 · source only']:.3f})")

### 7.4 Step 3 — Denoised pseudo-labels with a mean teacher

Now use the unlabelled target images themselves:

- The **student** is the network being trained. The **teacher** has the same architecture, but its weights are an
  **exponential moving average** (EMA) of the student's, $\theta_{teacher} \leftarrow 0.99\,\theta_{teacher} +
  0.01\,\theta_{student}$ after every step — a slower, smoother, more reliable copy (Tarvainen & Valpola, 2017).
- The teacher looks at a **bilateral-filtered** target image. A bilateral filter averages each pixel with neighbours that
  are both *near in space* and *similar in intensity*:
  $$v'_p=\frac{\sum_{q}G_{\sigma_d}(\lVert p-q\rVert)\,G_{\sigma_r}(v_q-v_p)\,v_q}{\sum_{q}G_{\sigma_d}(\lVert p-q\rVert)\,G_{\sigma_r}(v_q-v_p)},$$
  so it smooths noise but stops at edges. (Vox-UDA's *improved* bilateral filter replaces the intensity difference
  $v_q - v_p$ by a difference of local Laplacian-gradient vectors; we use the standard filter from scikit-image.)
- Only pixels where the teacher is **confident** ($p > \eta$ or $p < 1-\eta$, $\eta = 0.85$) become pseudo-labels for the
  student, which sees the **raw** noisy target image. Meanwhile the student keeps training on noise-injected source images.

In [ ]:
# ── 7.4 Bilateral-filtered targets and mean-teacher self-training ──
def bilateral(img: np.ndarray) -> np.ndarray:
    """Standard bilateral filter (scikit-image) on an image rescaled to [0, 1]; returns a normalised image."""
    lo, hi = img.min(), img.max()
    return normalize(denoise_bilateral((img - lo) / (hi - lo), win_size=7, sigma_color=0.1, sigma_spatial=2))


t0 = time.time()
Xt_seg_bil = np.stack([bilateral(x) for x in Xt_seg]).astype(np.float32)
print(f"bilateral-filtered {len(Xt_seg_bil)} target images in {time.time() - t0:.1f} s")


def mean_teacher(student, X_src, M_src, X_tgt, X_tgt_teacher, noise_sigma, steps=150, eta=0.85, ema=0.99,
                 lr=5e-4, seed=SEED, monitor=None):
    """Vox-UDA-style DPL: EMA teacher labels (denoised) target images; confident pixels supervise the student."""
    torch.manual_seed(seed)
    student = copy.deepcopy(student).to(DEVICE).train()
    teacher = copy.deepcopy(student)
    for p in teacher.parameters():
        p.requires_grad_(False)
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    Xs_, Ms_, Xt_, Xtt_ = as_tensor(X_src), as_tensor(M_src), as_tensor(X_tgt), as_tensor(X_tgt_teacher)
    history = []
    for step in range(steps):
        bs, bt = torch.randint(0, len(X_src), (32,)), torch.randint(0, len(X_tgt), (32,))
        loss_src = F.binary_cross_entropy_with_logits(student(Xs_[bs] + noise_sigma * torch.randn_like(Xs_[bs])), Ms_[bs])
        with torch.no_grad():
            p_teacher = torch.sigmoid(teacher(Xtt_[bt]))
        confident = ((p_teacher > eta) | (p_teacher < 1 - eta)).float()
        pseudo = (p_teacher > 0.5).float()
        loss_tgt = (F.binary_cross_entropy_with_logits(student(Xt_[bt]), pseudo, reduction="none")
                    * confident).sum() / confident.sum().clamp(min=1)
        opt.zero_grad(); (loss_src + loss_tgt).backward(); opt.step()
        with torch.no_grad():                                           # EMA update of the teacher
            for pt, ps in zip(teacher.parameters(), student.parameters()):
                pt.mul_(ema).add_(ps, alpha=1 - ema)
        if monitor is not None and (step + 1) % 25 == 0:
            history.append((step + 1, dice(teacher, *monitor), float(confident.mean())))
            student.train()
    return teacher.eval(), history


t0 = time.time()
seg_dpl, hist_dpl = mean_teacher(seg_ngm, Xs_seg, Ms_seg, Xt_seg, Xt_seg_bil, sigma_t,
                                 monitor=(Xt_seg_test, Mt_seg_test))
SEG["3 · + denoised pseudo-labels"] = dice(seg_dpl, Xt_seg_test, Mt_seg_test)
print(f"mean-teacher training in {time.time() - t0:.1f} s")
for step, d, frac in hist_dpl:
    print(f"  step {step:3d}: teacher Dice on target {d:.3f} | confident pixels {100 * frac:.0f} %")

In [ ]:
# ── 7.5 Step by step: what each component buys ──
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), gridspec_kw={"width_ratios": [1, 1.6]})
names, vals = list(SEG), list(SEG.values())
axes[0].bar(range(len(vals)), vals, color=["tab:gray", "tab:blue", "tab:green"])
for i, v in enumerate(vals):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=8)
axes[0].set_xticks(range(len(vals))); axes[0].set_xticklabels([n.split("·")[1].strip() for n in names], fontsize=7, rotation=15)
axes[0].set_ylim(min(vals) - 0.1, 1.0); axes[0].set_ylabel("Dice on target test"); axes[0].set_title("Vox-UDA-lite, step by step")
k = 3
show = [Xt_seg_test[:k], Mt_seg_test[:k]]
with torch.no_grad():
    for m in [seg_src, seg_ngm, seg_dpl]:
        show.append((torch.sigmoid(m(as_tensor(Xt_seg_test[:k]))) > 0.5).float().cpu().numpy()[:, 0])
grid = np.concatenate([np.concatenate(list(s), 0) for s in show], 1)
axes[1].imshow(grid); axes[1].axis("off")
axes[1].set_title("columns: target image | truth | source only | + NGM | + DPL", fontsize=8)
plt.tight_layout(); plt.show()
assert SEG["2 · + noise generation"] > SEG["1 · source only"], "noise generation should help on the noisy target"

🔍 **What just happened?** The source-only network, which never saw heavy noise, scatters false positives over the
noisy background and misses parts of molecules. Simply **measuring** the target noise and injecting it into source
images closes most of the gap — no target label needed, and only ten target images were used for the estimate. The
mean teacher then lets the unlabelled target images speak for themselves; its gain here is small and can drift if you
train too long (pseudo-label errors reinforce themselves — "confirmation bias"), which is why the confidence threshold
$\eta$ and the slow EMA teacher matter.

✏️ **Your turn.** (a) Re-run 7.4 with `X_tgt_teacher=Xt_seg` (no bilateral filter for the teacher). (b) Set `eta=0.6`
(accept less confident pseudo-labels). (c) Train for `steps=400`. Which change hurts, and why?

### 📄 Paper connection — Vox-UDA (Li *et al.*, AAAI 2025)

- **Paper**: Li H, Li X, Shi J, Chen H, Du B, Kihara D, Barthelemy J, Shen J, Xu M. *Vox-UDA: Voxel-wise Unsupervised
  Domain Adaptation for Cryo-Electron Subtomogram Segmentation with Denoised Pseudo-Labeling.* AAAI 2025, pp. 406–414.
  [doi:10.1609/aaai.v39i1.32019](https://doi.org/10.1609/aaai.v39i1.32019), [arXiv:2406.18610](https://arxiv.org/abs/2406.18610).
  Extension: *From Unsupervised to Zero-Shot: 3D Domain Adaptation for Cryo-Electron Tomography Segmentation*, Int. J.
  Comput. Vis. 134(5), 2026, [doi:10.1007/s11263-026-02802-6](https://doi.org/10.1007/s11263-026-02802-6) (not summarised here).
- **Question it answers**: voxel-wise segmentation networks trained on labelled simulated subtomograms fail on real ones;
  adapt them with **no** target labels.
- **What the paper actually does**: a mean-teacher UDA framework on a **VoxResNet** backbone with (1) the **NGM** —
  high-pass target samples in Fourier space (retention rate ρ = 24.4 %), average over $N_{sampled}=10$, take the variance
  $\sigma_t^2$, inject $\mathcal N(0,\sigma_t^2 I)$ into source samples, plus the feature-consistency loss above; (2)
  **DPL with an improved bilateral filter** (3 × 3 × 3 window, $\sigma_d = 120$, $\sigma_r = 1.2$, range kernel on
  Laplacian-gradient differences) applied to target inputs before the EMA teacher predicts; labels with confidence above
  $\eta = 0.85$ supervise the student. Total loss $\mathcal L=\mathcal L_{seg}+\mathcal L_{con}+\mathcal L_{dis}$, where
  $\mathcal L_{dis}$ is an adversarial domain-discriminator loss. Training: Adam, learning rate 10⁻³, 300 epochs, batch 16.
- **What our toy keeps**: the noise estimate from high-passed target images, noise injection with a consistency loss, a
  bilateral-filtered EMA teacher with an η = 0.85 confidence threshold.
- **What our toy leaves out**: the improved bilateral filter, the adversarial discriminator loss, VoxResNet, 3D data;
  our 1/ρ variance correction is our own choice (see 7.3).
- **Result reported**: source data were simulated subtomograms (two groups of three complexes, SNR 0.03 and 0.05); the
  target was the real Poly-GA neuron dataset (66 26S proteasome, 66 TRiC and 901 ribosome subtomograms). On the first
  source group the authors report 50.3 mIoU / 65.9 Dice for Vox-UDA vs 41.5 / 56.8 for the best baseline (LE-UDA) and
  9.7 / 17.4 for source-only. They also report a "fully supervised" reference of 46.0 / 61.6 — *below* Vox-UDA, which
  is worth discussing critically (🧠 question 4).
- **Code**: no link in the arXiv text.

---
## 8 · Summary: which tool for which situation?

In [ ]:
# ── 8.1 All classification results side by side ──
fig, ax = plt.subplots(figsize=(7.5, 3.4))
names, vals = list(RESULTS), list(RESULTS.values())
colors = ["tab:gray"] + ["tab:blue"] * (len(vals) - 1)
ax.barh(range(len(vals)), vals, color=colors)
for i, v in enumerate(vals):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
ax.axvline(acc_src_src, color="k", ls="--", lw=1)
ax.text(acc_src_src, len(vals) - 0.4, " source-only on SOURCE", fontsize=7, va="bottom", ha="right")
ax.set_yticks(range(len(vals))); ax.set_yticklabels(names, fontsize=8); ax.invert_yaxis()
ax.set_xlim(0.25, 1.0); ax.set_xlabel("accuracy on the target test set (chance = 0.25)")
ax.set_title("Closing the simulation-to-reality gap (one toy, one seed)")
plt.tight_layout(); plt.show()
for k_, v in RESULTS.items():
    print(f"  {k_:28s} {v:.3f}")

🔍 **Reading the chart.** Every method recovers part of the gap between the dashed line (how good the network is on the
data it was trained for) and the grey bar (source-only on the target). Exact rankings in a toy with one random seed are
**not** reliable — re-run with `SEED = 1` and some bars will swap. What is reliable is the *menu*:

| Situation | Tool | Needs target images? | Needs target labels? | Lab paper |
|---|---|---|---|---|
| You can guess the range of imaging conditions | domain randomization | no | no | Che 2019 ⚠️, Cryo-Shift 2022 |
| You have unlabelled real images | adversarial feature alignment (DANN / ADDA) | yes | no | 3D-ADA 2019, Cryo-Shift 2022 |
| An expert can label ~3–7 per class | few-shot DA (CORAL + pair discriminator) | yes | a few | Yu 2021 |
| You want realistic *training data* for any model | unpaired translation (CycleGAN/WGAN) | yes | no | CryoETGAN 2022 |
| Voxel-wise segmentation, noise is the main gap | noise generation + denoised pseudo-labels | yes | no | Vox-UDA 2025 |
| Robustness to worst-case perturbations | adversarial training | no | no | RAT 2019 ⚠️ |

💡 **The biology lesson.** A model's accuracy is a property of the model **and** of the data it is applied to. When a paper
reports "95 % accuracy", always ask: *on which domain?* — simulated, purified single-particle, or cellular — and whether
the test data came from the same microscope, sample and session as the training data (Module 05's lesson on data
leakage, now one level up).

---
## ✅ Checkpoint

1. In one sentence each, what are the *source* and the *target* domain in the cryo-ET setting, and what is covariate shift?
   <details><summary>Answer</summary>Source = labelled simulated (or other-microscope) subtomograms; target = the real,
   unlabelled subtomograms we care about. Covariate shift: the images' distribution P(x) differs, but the labelling rule
   P(y|x) ("this shape is a ribosome") is the same.</details>
2. What does a gradient-reversal layer do in the forward and in the backward pass, and why is that useful?
   <details><summary>Answer</summary>Forward: nothing (identity). Backward: multiplies the gradient by −λ. The discriminator
   is trained to tell domains apart while the feature extractor receives the opposite gradient and learns to hide the
   domain — the min–max game in a single backward pass.</details>
3. 3D-ADA and DANN both use a domain discriminator. Name one difference.
   <details><summary>Answer</summary>3D-ADA is ADDA-style: two stages, a separate target extractor initialised from the
   frozen source extractor and trained with an inverted-label loss; DANN trains one shared extractor and the classifier
   jointly with a gradient-reversal layer.</details>
4. Why can domain randomization fail even with millions of simulated images?
   <details><summary>Answer</summary>If the real data contain something the randomizer never produces (e.g. a noise
   colour, crowding or conformations outside the sampled ranges), the target is still outside the training distribution.</details>
5. In Vox-UDA's noise generation module, why look at *high* frequencies to estimate the noise?
   <details><summary>Answer</summary>At tomographic resolution a molecule's signal lives mostly at low frequencies, so the
   high-frequency part of a target image is dominated by noise; its variance (scaled by the fraction of coefficients
   kept) estimates the noise variance without any clean target image.</details>
6. Why does a mean teacher use an *exponential moving average* of the student's weights?
   <details><summary>Answer</summary>The average changes slowly and smooths out the student's step-to-step noise, so its
   pseudo-labels are more stable and less prone to reinforcing the student's latest mistakes.</details>

## 🧠 Questions & discussion

1. DANN aligns the *marginal* feature distributions. Construct an example (class proportions!) where perfect alignment
   makes target accuracy **worse**. How do Cryo-Shift's class-wise discriminators help?
2. We monitored target accuracy during adaptation using the hidden labels. In a real study, how would you choose λ and
   when to stop, without any target labels?
3. A learned simulator (Section 6) could also *hallucinate* features that real data do not have. How could that bias a
   downstream biological conclusion (e.g. abundance of a complex)? How would you test for it?
4. Vox-UDA reports a "fully supervised" reference **below** its unsupervised result. Give two possible explanations (think
   about how much labelled real data exist, and about noise in the labels).
5. Domain shift also exists *between* real datasets (two microscopes, two cell types, two labs). Which method in this
   notebook would you try first when moving a ribosome picker from *M. pneumoniae* to neurons, and why?

## 🚀 Bonus — going further

- **ADDA instead of DANN.** Implement 3D-ADA's two-stage scheme: freeze `model_src`, copy its `features` into a target
  extractor, and train it against a discriminator with the inverted-label loss. Compare stability with DANN.
- **Class-conditional alignment.** Give DANN one discriminator per class, weighted by the classifier's softmax probability
  (Cryo-Shift's $\mathcal L_{fd}$), and test with *unequal* class proportions in the target.
- **Go 3D.** Replace `random_projections` with full 24³ subtomograms (`rotate_volume` + 3D wedge) and the CNN with a 3D
  one from Module 15; budget ~10× more time.
- **Real data.** Try a simulated-to-real transfer on a public dataset (e.g. EMPIAR-10499 ribosomes, used by several
  Xu-lab papers) with the official AITom code: `aitom/classify/deep/unsupervised/cryo_shift/` and
  `aitom/classify/deep/supervised/cnn/domain_adaptation_adversarial/`.
- **Improved bilateral filter.** Implement Vox-UDA's IBF (range kernel on Laplacian-gradient differences) and compare the
  teacher's pseudo-label accuracy with the standard bilateral filter.

## 📝 Lab note

```text
Module 18 — date:
Input: 2D projections of 4 zoo molecules; source microscope (simulator) vs target microscope (different CTF, wedge,
       coloured noise, neighbours); segmentation: SNR 20 → SNR 0.1
Output: target accuracy / Dice per adaptation method
Method: source-only, domain randomization, DANN (gradient reversal), randomization + DANN, few-shot + CORAL,
        WGAN-learned microscope, Vox-UDA-lite (NGM + mean teacher on bilateral-filtered targets)
Parameter I changed and what happened:
A failure mode / surprise:
Paper link: Lin 2019 (3D-ADA); Bandyopadhyay 2022 (Cryo-Shift); Yu 2021; Wu 2022 (CryoETGAN); Li 2025 (Vox-UDA)
```

## 📚 References

1. Lin R, Zeng X, Kitani K, Xu M. Adversarial domain adaptation for cross data source macromolecule in situ structural classification in cellular electron cryo-tomograms. *Bioinformatics* 35(14):i260–i268 (2019). doi:10.1093/bioinformatics/btz364
2. Che C, Xian Z, Zeng X, Gao X, Xu M. Domain randomization for macromolecule structure classification and segmentation in electron cyro-tomograms. *IEEE BIBM* 2019, pp. 6–11. ⚠️ content not verified.
3. Wu X, Mao Y, Wang H, Zeng X, Gao X, Xing E, Xu M. Regularized adversarial training (RAT) for robust cellular electron cryo tomograms classification. *IEEE BIBM* 2019. ⚠️ content not verified.
4. Yu L, Li R, Zeng X, Wang H, Jin J, Yang G, Jiang R, Xu M. Few shot domain adaptation for in situ macromolecule structural classification in cryoelectron tomograms. *Bioinformatics* 37(2):185–191 (2021). doi:10.1093/bioinformatics/btaa671
5. Bandyopadhyay H, Deng Z, Ding L, Liu S, Uddin MR, Zeng X, Behpour S, Xu M. Cryo-shift: reducing domain shift in cryo-electron subtomograms with unsupervised domain adaptation and randomization. *Bioinformatics* 38(4):977–984 (2022). doi:10.1093/bioinformatics/btab794
6. Wu X, Li C, Zeng X, Wei H, Deng HW, Zhang J, Xu M. CryoETGAN: Cryo-electron tomography image synthesis via unpaired image translation. *Front Physiol* 13:760404 (2022). doi:10.3389/fphys.2022.760404
7. Li H, Li X, Shi J, Chen H, Du B, Kihara D, Barthelemy J, Shen J, Xu M. Vox-UDA: Voxel-wise unsupervised domain adaptation for cryo-electron subtomogram segmentation with denoised pseudo-labeling. *AAAI* 39(1):406–414 (2025). doi:10.1609/aaai.v39i1.32019
8. Li H *et al.* From unsupervised to zero-shot: 3D domain adaptation for cryo-electron tomography segmentation. *Int J Comput Vis* 134(5) (2026). doi:10.1007/s11263-026-02802-6
9. Ganin Y, Lempitsky V. Unsupervised domain adaptation by backpropagation. *ICML* 2015 (DANN, gradient reversal).
10. Tzeng E, Hoffman J, Saenko K, Darrell T. Adversarial discriminative domain adaptation. *CVPR* 2017 (ADDA).
11. Ben-David S, Blitzer J, Crammer K, Kulesza A, Pereira F, Vaughan JW. A theory of learning from different domains. *Mach Learn* 79:151–175 (2010).
12. Sun B, Saenko K. Deep CORAL: correlation alignment for deep domain adaptation. *ECCV Workshops* 2016.
13. Zhu JY, Park T, Isola P, Efros AA. Unpaired image-to-image translation using cycle-consistent adversarial networks. *ICCV* 2017 (CycleGAN).
14. Arjovsky M, Chintala S, Bottou L. Wasserstein generative adversarial networks. *ICML* 2017; Miyato T *et al.* Spectral normalization for GANs. *ICLR* 2018.
15. Tarvainen A, Valpola H. Mean teachers are better role models. *NeurIPS* 2017.
16. Tobin J *et al.* Domain randomization for transferring deep neural networks from simulation to the real world. *IROS* 2017.

## ➡️ Next

**[19 · Learning to align without labels: Gum-Net and equivariant transformers](19_deep_alignment.ipynb)** — the
networks so far *classified* particles; next they learn to *align* them, the step every subtomogram average depends on.
Course overview: [`README.md`](../README.md) · glossary: [`GLOSSARY.md`](../GLOSSARY.md).